# SNP Allele Frequency

This tutorial focuses on allele-frequency APIs for `SNPObject`, including grouped and ancestry-specific frequencies plus chunked streaming. The genotype input comes from `build_synthetic_snp_dataset()`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from snputils import allele_freq_stream
from snputils.datasets import build_synthetic_snp_dataset

RESULTS_DIR = Path("results/tutorials")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SEED = 20240520

rng = np.random.default_rng(SEED)
snp = build_synthetic_snp_dataset(
    n_samples=6,
    n_snps=9,
    seed=SEED,
    n_populations=3,
    missing_rate=0.04,
    phased=True,
    variant_prefix="rs_af",
)
labels = snp.sample_fid
snp.calldata_lai = rng.integers(0, 2, size=snp.genotypes.shape, dtype=np.int8)
snp.ancestry_map = {"0": "ANC0", "1": "ANC1"}
snp

SNPObject(shape=(9, 6, 2), n_snps=9, n_samples=6, genotypes_shape=(9, 6, 2), calldata_lai_shape=(9, 6, 2), calldata_gp_shape=None, has_variant_metadata=True, has_ancestry_map=True)

## Cohort allele frequency

With no labels, `allele_freq()` counts called haplotypes across all samples for each SNP.

In [2]:
af, called = snp.allele_freq(return_counts=True)
pd.DataFrame({"variant": snp.variants_id, "af": af, "called_haplotypes": called})

,variant,af,called_haplotypes
0,rs_af_00001,0.909091,11
1,rs_af_00002,0.583333,12
2,rs_af_00003,0.500000,12
3,rs_af_00004,0.818182,11
4,rs_af_00005,0.833333,12
5,rs_af_00006,0.200000,10
6,rs_af_00007,0.909091,11
7,rs_af_00008,0.454545,11
8,rs_af_00009,0.750000,12


## Grouped allele frequency

Pass `sample_labels` to compute one column per group. Missing genotypes are excluded from both numerator and denominator.

In [3]:
af_by_group, called_by_group = snp.allele_freq(
    sample_labels=labels,
    return_counts=True,
    as_dataframe=True,
)

display(af_by_group)
display(called_by_group)

,POP_A,POP_B,POP_C
0,0.75,1.00,1.000000
1,0.75,0.50,0.500000
2,0.75,0.50,0.250000
3,0.75,0.75,1.000000
4,0.75,1.00,0.750000
5,0.00,0.25,0.333333
6,1.00,0.75,1.000000
7,0.50,0.50,0.333333
8,1.00,0.25,1.000000


,POP_A,POP_B,POP_C
0,4,3,4
1,4,4,4
2,4,4,4
3,4,4,3
4,4,4,4
5,3,4,3
6,3,4,4
7,4,4,3
8,4,4,4


## Ancestry-specific allele frequency

When `calldata_lai` is present, `ancestry=` restricts the count to haplotypes assigned to that ancestry code.

In [4]:
af_anc1, called_anc1 = snp.allele_freq(ancestry=1, return_counts=True)
pd.DataFrame({"variant": snp.variants_id, "anc1_af": af_anc1, "called_anc1_haplotypes": called_anc1})

,variant,anc1_af,called_anc1_haplotypes
0,rs_af_00001,1.000000,6
1,rs_af_00002,0.833333,6
2,rs_af_00003,0.400000,5
3,rs_af_00004,0.750000,8
4,rs_af_00005,1.000000,5
5,rs_af_00006,0.250000,8
6,rs_af_00007,1.000000,5
7,rs_af_00008,0.250000,4
8,rs_af_00009,0.666667,3


## Streaming allele frequency

`allele_freq_stream()` processes SNP chunks and returns the same result as eager computation for in-memory objects. Use it when the source object or reader is too large to handle in one block.

In [5]:
stream_af, stream_called = allele_freq_stream(
    snp,
    chunk_size=4,
    sample_labels=labels,
    ancestry=1,
    return_counts=True,
)

eager_af, eager_called = snp.allele_freq(
    sample_labels=labels,
    ancestry=1,
    return_counts=True,
)

print(np.allclose(stream_af, eager_af, equal_nan=True))
print(np.array_equal(stream_called, eager_called))

True
True
